# Unidad: Redes Neuronales Artificiales
## Redes Neuronales con Keras — Clasificación de dígitos MNIST
### Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER

<!-- <img src="https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-mikhail-nilov-9301821.jpg" alt="Red neuronal con Keras" width="700"/>

*Foto de [Mikhail Nilov](https://www.pexels.com/@mikhail-nilov/) en [Pexels](https://www.pexels.com/)* -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/ann/02_ANN_Keras_MNIST.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook vas a poder:

✅ Entender la API de **Keras** para construir redes neuronales densas (MLP)  
✅ Preprocesar imágenes para una red neuronal: aplanar, normalizar, codificar etiquetas  
✅ Diseñar y entrenar un MLP de 3 capas sobre el dataset **MNIST**  
✅ Aplicar **Dropout** como técnica de regularización para reducir overfitting  
✅ Evaluar el modelo con métricas de accuracy y visualizar la curva de entrenamiento  

---

## 📖 Marco Teórico

### El dataset MNIST

**MNIST** (*Modified National Institute of Standards and Technology*) es el dataset de referencia del deep learning. Contiene **70,000 imágenes** de dígitos escritos a mano (0–9), en escala de grises, de 28×28 píxeles:

- **60,000** imágenes de entrenamiento
- **10,000** imágenes de prueba

Cada imagen se representa como una matriz de 28×28 valores enteros (0–255). Para una red densa, se aplana a un vector de **784 elementos**.

### Keras y la API Sequential

**Keras** es una API de alto nivel para construir y entrenar redes neuronales. Actualmente está integrada en **TensorFlow 2**. La forma más sencilla de definir una red es con la clase `Sequential`:

```python
model = Sequential()
model.add(Dense(256, activation='relu', input_dim=784))
model.add(Dropout(0.45))
model.add(Dense(10, activation='softmax'))
```

### Capas clave

| Capa | Función |
|---|---|
| `Dense(n, activation)` | Capa densa (fully connected) con $n$ neuronas |
| `Dropout(rate)` | Desactiva aleatoriamente una fracción de neuronas durante el entrenamiento |
| Activación `relu` | $\max(0, z)$ — capas ocultas |
| Activación `softmax` | Distribución de probabilidad sobre $k$ clases — capa de salida |

### One-Hot Encoding

Las etiquetas deben convertirse de escalar a vector binario para la función de pérdida `categorical_crossentropy`:

$$y = 3 \quad \Rightarrow \quad \mathbf{y} = [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]$$

### Dropout — Regularización

El **Dropout** desactiva aleatoriamente una fracción $p$ de neuronas en cada paso de entrenamiento:

$$\tilde{h}_i = h_i \cdot \text{Bernoulli}(1 - p)$$

Con `dropout=0.45`, solo el $55\%$ de las neuronas participan en cada actualización. Esto obliga a la red a aprender representaciones más robustas y reduce el overfitting.

> 📌 **Clave**: el Dropout se desactiva automáticamente durante la inferencia (`model.predict`). No afecta al modelo final, solo al entrenamiento.

### Función de pérdida y optimizador

| Elemento | Elección | Cuándo usarla |
|---|---|---|
| `categorical_crossentropy` | Clasificación multiclase con one-hot | Salida softmax, $>2$ clases |
| `sparse_categorical_crossentropy` | Ídem, sin one-hot | Etiquetas como enteros |
| `binary_crossentropy` | Clasificación binaria | Salida sigmoid |

| `adam` | Optimizador adaptativo | Por defecto en la mayoría de problemas |

## 📦 Paso 1: Imports y configuración

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"✅ TensorFlow {tf.__version__} | Keras {keras.__version__}")
print(f"💡 GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 🔢 Paso 2: Carga y exploración del dataset MNIST

In [ ]:
# Carga del dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")
print(f"Clases: {np.unique(y_train)} ({len(np.unique(y_train))} dígitos)")
print(f"Rango de valores: [{X_train.min()}, {X_train.max()}]")

In [ ]:
# Visualización de muestras del dataset
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"[{y_train[i]}]", fontsize=9)
    ax.axis('off')
plt.suptitle("Muestras del dataset MNIST (primeras 20 imágenes)", y=1.02)
plt.tight_layout()
plt.show()

## 🔧 Paso 3: Preprocesamiento

In [ ]:
# 1. Aplanar (flatten): 28x28 -> 784
X_train_flat = X_train.reshape(-1, 784).astype('float32')
X_test_flat  = X_test.reshape(-1, 784).astype('float32')

# 2. Normalizar: [0,255] -> [0,1]
X_train_norm = X_train_flat / 255.0
X_test_norm  = X_test_flat  / 255.0

# 3. One-hot encoding de las etiquetas
NUM_CLASSES = 10
y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  NUM_CLASSES)

print(f"X_train normalizado: {X_train_norm.shape}, rango [{X_train_norm.min():.1f}, {X_train_norm.max():.1f}]")
print(f"y_train one-hot:     {y_train_cat.shape}")
print(f"Ejemplo etiqueta 0:  {y_train[0]} → {y_train_cat[0]}")

## 🏗️ Paso 4: Construcción del modelo

La arquitectura que usaremos es una red densa totalmente conectada (*Fully Connected / Dense Network*):

| Capa | Tipo | Neuronas | Activación |
|------|------|----------|------------|
| Entrada | Dense | 512 | ReLU |
| Regularización | Dropout | — | 20% |
| Oculta | Dense | 256 | ReLU |
| Regularización | Dropout | — | 20% |
| Salida | Dense | 10 | Softmax |

- **ReLU** (*Rectified Linear Unit*): activa neuronas con valores positivos → resuelve el problema del gradiente desvaneciente.  
- **Dropout**: desactiva aleatoriamente un porcentaje de neuronas en cada paso → regularización, evita sobreajuste.  
- **Softmax**: convierte la salida en probabilidades que suman 1 → ideal para clasificación multiclase.

In [ ]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(784,), name='capa_entrada'),
    Dropout(0.2, name='dropout_1'),
    Dense(256, activation='relu', name='capa_oculta'),
    Dropout(0.2, name='dropout_2'),
    Dense(NUM_CLASSES, activation='softmax', name='capa_salida')
], name='ANN_MNIST')

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 🏋️ Paso 5: Entrenamiento

In [ ]:
EPOCHS     = 20
BATCH_SIZE = 128

history = model.fit(
    X_train_norm, y_train_cat,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_split= 0.1,
    verbose         = 1
)

print(f"\n✅ Entrenamiento completado — {EPOCHS} épocas")

## 📉 Paso 6: Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Pérdida ---
axes[0].plot(history.history['loss'],     label='Entrenamiento', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validación',    linewidth=2, linestyle='--')
axes[0].set_title('Pérdida (Loss)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# --- Accuracy ---
axes[1].plot(history.history['accuracy'],     label='Entrenamiento', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validación',    linewidth=2, linestyle='--')
axes[1].set_title('Exactitud (Accuracy)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Épocas')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — ANN sobre MNIST', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 📊 Paso 7: Evaluación y matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Evaluación sobre test set
test_loss, test_acc = model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f"✅ Test Loss:     {test_loss:.4f}")
print(f"✅ Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
# Predicciones y matriz de confusión
y_pred_proba = model.predict(X_test_norm, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicho',  fontsize=12)
plt.ylabel('Real',      fontsize=12)
plt.title('Matriz de Confusión — ANN sobre MNIST', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## 🎓 Resumen y Conclusiones

En este notebook entrenamos una **Red Neuronal Artificial (ANN)** densa para clasificar los dígitos escritos a mano del dataset **MNIST** usando **Keras / TensorFlow**.

### Resultados obtenidos

| Métrica | Valor |
|---------|-------|
| Accuracy en test | ~98 % |
| Loss en test | ~0.07 |

### Puntos clave

- **Arquitectura**: red totalmente conectada con capas Dense + Dropout.  
- **Preprocesamiento**: normalización de píxeles a `[0, 1]` y codificación one-hot de etiquetas.  
- **Optimizador Adam**: ajuste adaptativo del learning rate, converge rápidamente.  
- **Dropout (20%)**: regularización que reduce el sobreajuste sin aumentar los datos.  
- **Softmax + Categorical Crossentropy**: combinación estándar para clasificación multiclase.

### Próximos pasos

- Usar **Redes Convolucionales (CNN)** para capturar patrones espaciales y superar el 99 % de accuracy.  
- Aplicar **Data Augmentation** (rotaciones, traslaciones) para mejorar la generalización.  
- Explorar arquitecturas más profundas y técnicas como **Batch Normalization**.

---
> 💡 La ANN densa es un buen punto de partida para visión por computadora, pero las **CNN** aprovechan la estructura espacial de las imágenes y logran resultados superiores con menos parámetros.

---
<br>

**Inteligencia Artificial** · Licenciatura en Sistemas · FCAD - UNER  
Autor: Cristian Pacífico · [GitHub](https://github.com/CristianPacifico/ia-ls-fcad-uner)  
Licencia: [MIT](https://opensource.org/licenses/MIT)